In [ ]:
#include <cstddef>
#include <vector>
#include <random>
#include <iostream>

#include "xsimd/xsimd.hpp"

Comparing two implementation of element wise mean.

In [ ]:
std::vector<double> random_vector(std::size_t n)
{
    static std::mt19937 gen(42);
    std::uniform_real_distribution<double> dist(0.0, 1.0);
    std::vector<double> v(n);
    for (auto& x : v)
    {
        x = dist(gen);
    }
    return v;
}

In [ ]:
std::vector<double> mean_scalar(const std::vector<double>& a, const std::vector<double>& b)
{
    std::size_t size = a.size();
    std::vector<double> res(size);
    for (std::size_t i = 0; i < size; ++i)
    {
        res[i] = (a[i] + b[i]) / 2;
    }
    return res;
}

In [ ]:
std::vector<double> mean_simd(const std::vector<double>& a, const std::vector<double>& b)
{
    using b_type = xsimd::batch<double>;
    std::size_t inc = b_type::size;
    std::size_t size = a.size();
    std::vector<double> res(size);

    // size for which the vectorization is possible
    std::size_t vec_size = size - size % inc;
    for (std::size_t i = 0; i < vec_size; i += inc)
    {
        b_type avec = b_type::load_unaligned(&a[i]);
        b_type bvec = b_type::load_unaligned(&b[i]);
        b_type rvec = (avec + bvec) / 2;
        rvec.store_unaligned(&res[i]);
    }
    // Remaining part that cannot be vectorize
    for (std::size_t i = vec_size; i < size; ++i)
    {
        res[i] = (a[i] + b[i]) / 2;
    }
    return res;
}

In [ ]:
// Print v[begin:end] as grayscale ASCII blocks (dark = 0, light = 1).
void print_grayscale(const std::vector<double>& v, std::size_t begin, std::size_t end)
{
    for (std::size_t i = begin; i < end; ++i)
    {
        // map value in [0, 1] to a 24-bit truecolor gray (256 levels)
        int gray = static_cast<int>(v[i] * 255);
        std::cout << "\033[38;2;" << gray << ";" << gray << ";" << gray << "m█";
    }
    std::cout << "\033[0m\n";
}

In [ ]:
std::size_t n = 1000;
std::vector<double> const a = random_vector(n);
std::vector<double> const b = random_vector(n);

In [ ]:
auto const res_simd = mean_simd(a, b);

In [ ]:
auto const res_scalar = mean_scalar(a, b);

Each block is a value in `[0, 1]` shown as a grayscale shade (dark = low, light = high). The vector is drawn in chunks of 100, each chunk stacking input `a`, the SIMD mean, the scalar mean, and input `b`. The two mean rows should look identical, each sitting between its inputs.

In [ ]:
std::size_t const width = 100;
for (std::size_t begin = 0; begin < n; begin += width)
{
    std::size_t end = std::min(begin + width, n);
    std::cout << "a:      ";
    print_grayscale(a, begin, end);
    std::cout << "simd:   ";
    print_grayscale(res_simd, begin, end);
    std::cout << "scalar: ";
    print_grayscale(res_scalar, begin, end);
    std::cout << "b:      ";
    print_grayscale(b, begin, end);
    std::cout << "\n";
}